In [ ]:
# --- ensure repo-root cwd so data/ paths resolve from anywhere ---
import os, sys
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src'))

# SoH estimation & RUL forecasting — baseline

Per-vehicle, per-month feature table from both feeds for the most-aged overlapping cohort, with
**coulomb-counting SoH** (intellicar) as the target. Trains **GPR** (uncertainty) and **LightGBM**
(importance) with leakage-safe grouping by VIN, then an **empirical capacity-fade** RUL forecast.

Coulomb counting uses the vectorized `src/soh.py` (GPU via cuDF if RAPIDS is installed, else fast
vectorized CPU).

In [ ]:
import numpy as np, pandas as pd, time
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
import soh

IC, MH = "data/mahindra/intellicar", "data/mahindra/feed"
GAP_S = soh.GAP_S

t0 = time.time()
ic = ds.dataset(IC, format="parquet").to_table(
        columns=["vin", "eventAt", "soc", "current", "batteryVoltage"]).to_pandas()
ic["t"] = pd.to_datetime(pd.to_numeric(ic["eventAt"], errors="coerce"), unit="ms")
for c in ["soc", "current", "batteryVoltage"]:
    ic[c] = pd.to_numeric(ic[c], errors="coerce")
ic = ic.dropna(subset=["vin", "t", "soc", "current"]).query("0 <= soc <= 100")
ic["month"] = ic["t"].dt.to_period("M").dt.to_timestamp()
print(f"intellicar: {len(ic):,} rows, {ic['vin'].nunique()} VINs, loaded in {time.time()-t0:.1f}s")

## 1. SoH target — vectorized coulomb counting

In [ ]:
t0 = time.time()
cap_month, used_gpu = soh.coulomb_capacity_monthly(ic[["vin", "t", "soc", "current"]], backend="auto")
_reg = pd.read_csv("data/manifests/mahindra_registration.csv")
_reg["reg"] = pd.to_datetime(_reg["vehicle_registration_date"], errors="coerce")
REG = dict(zip(_reg["vin"], _reg["reg"]))
target = soh.capacity_to_soh(cap_month, reg=REG)
print(f"coulomb counting ({'GPU/cuDF' if used_gpu else 'CPU vectorized'}) in {time.time()-t0:.1f}s")
print(f"SoH target: {len(target)} VIN-months across {target['vin'].nunique()} VINs; reg-anchored {int(target.groupby('vin')['used_reg'].first().sum())} VINs")
print(target.groupby('vin')['soh'].agg(['count', 'min']).round(1).to_string())

## 2. Features — electrical (intellicar) + usage/environment (mahindra feed)

In [ ]:
import features
t0 = time.time()
fe_ic, fgpu = features.electrical_features(
    ic[["vin", "t", "soc", "current", "batteryVoltage"]], backend="auto")
print(f"electrical features ({'GPU/cuDF' if fgpu else 'CPU vectorized'}) in {time.time()-t0:.1f}s -> {fe_ic.shape}")

mh = ds.dataset(MH, format="parquet").to_table(
        columns=["vin", "eventAt", "odometer", "distanceToEmpty", "latitude",
                 "longitude", "gearPosition", "batteryTemp"]).to_pandas()
mh["t"] = pd.to_datetime(pd.to_numeric(mh["eventAt"], errors="coerce"), unit="ms")
for c in ["odometer", "distanceToEmpty", "latitude", "longitude", "batteryTemp"]:
    mh[c] = pd.to_numeric(mh[c], errors="coerce")
mh = mh.dropna(subset=["vin", "t"]); mh["month"] = mh["t"].dt.to_period("M").dt.to_timestamp()
def usage_feats(g):
    odo = g["odometer"].dropna(); gp = g["gearPosition"].astype(str).str.upper()
    return pd.Series({"km_month": (odo.max() - odo.min()) if len(odo) else np.nan,
                      "odo_max": odo.max() if len(odo) else np.nan,
                      "temp_mean": g["batteryTemp"].mean(), "temp_max": g["batteryTemp"].max(),
                      "lat_mean": g["latitude"].mean(), "lon_mean": g["longitude"].mean(),
                      "frac_drive": gp.isin(["FORWARD", "REVERSE"]).mean(),
                      "dte_mean": g["distanceToEmpty"].mean()})
fe_mh = (mh.groupby(["vin", "month"], group_keys=True)[["odometer", "distanceToEmpty", "latitude",
                                                         "longitude", "gearPosition", "batteryTemp"]]
           .apply(usage_feats).reset_index())
print(f"intellicar features {fe_ic.shape}, mahindra features {fe_mh.shape}")

## 3. Assemble modeling table

In [ ]:
df = (target[["vin", "month", "soh", "capacity_ah", "age_months"]]
      .merge(fe_ic, on=["vin", "month"], how="left")
      .merge(fe_mh, on=["vin", "month"], how="left")
      .sort_values(["vin", "month"]).reset_index(drop=True))
df["cum_ah"] = df.groupby("vin")["ah_throughput"].cumsum()
df["cum_km"] = df.groupby("vin")["km_month"].cumsum()
df["wh_per_km"] = (df["ah_throughput"] * df["volt_mean"] / df["km_month"]).replace([np.inf, -np.inf], np.nan)

FEATURES = ["age_months", "cum_ah", "cum_km", "odo_max", "ah_throughput",
            "cur_abs_mean", "cur_abs_p95", "cur_dis_mean", "cur_chg_mean",
            "soc_mean", "frac_soc_high", "frac_soc_low", "volt_mean", "volt_min", "volt_max",
            "temp_mean", "temp_max", "lat_mean", "lon_mean", "frac_drive", "dte_mean"]
model_df = df.dropna(subset=["soh"]).copy()
Path("data/mahindra/features").mkdir(parents=True, exist_ok=True)
model_df.to_parquet("data/mahindra/features/feature_table.parquet", index=False)
print(f"Modeling table: {model_df.shape[0]} rows x {len(FEATURES)} features, {model_df['vin'].nunique()} VINs")
print("SoH range:", round(model_df['soh'].min(), 1), "-", round(model_df['soh'].max(), 1))

## 4. Estimation — GPR + LightGBM (grouped CV by VIN)

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb

X = model_df[FEATURES].to_numpy(); y = model_df["soh"].to_numpy(); groups = model_df["vin"]
gkf = GroupKFold(n_splits=min(5, model_df["vin"].nunique()))

def gpr():
    k = ConstantKernel(1.0) * RBF(length_scale=np.ones(len(FEATURES))) + WhiteKernel(1.0)
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         GaussianProcessRegressor(kernel=k, normalize_y=True, alpha=1e-3))

rows = []
for name, mk in [("GPR", gpr),
                 ("LightGBM", lambda: lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                                        num_leaves=15, min_child_samples=5, verbose=-1))]:
    preds = np.full(len(y), np.nan)
    for tr, te in gkf.split(X, y, groups):
        m = mk(); m.fit(X[tr], y[tr]); preds[te] = m.predict(X[te])
    rows.append({"model": name, "MAE": mean_absolute_error(y, preds), "R2": r2_score(y, preds)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

lgbm = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=15,
                         min_child_samples=5, verbose=-1).fit(X, y)
imp = pd.Series(lgbm.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nTop degradation-driver features (LightGBM importance):")
print(imp.head(10).to_string())

## 5. Forecasting baseline — empirical capacity fade -> RUL

In [ ]:
EOL = 80.0; rows = []
for vin, g in model_df.groupby("vin"):
    g = g.sort_values("age_months")
    if len(g) < 4:
        continue
    t = g["age_months"].to_numpy(); s = g["soh"].to_numpy(); mask = t > 0
    k = np.sum((100 - s[mask]) * np.sqrt(t[mask])) / np.sum(t[mask]) if mask.any() else np.nan
    t_eol = ((100 - EOL) / k) ** 2 if k and k > 0 else np.inf
    rows.append({"vin": vin, "soh_now": round(s[-1], 1), "age_now_mo": round(t[-1], 1),
                 "fade_k": round(k, 2), "t_eol_mo": round(t_eol, 1), "RUL_mo": round(max(t_eol - t[-1], 0), 1)})
rul = pd.DataFrame(rows).sort_values("RUL_mo")
print(rul.to_string(index=False))
Path("data/mahindra/soh").mkdir(parents=True, exist_ok=True)
rul.to_csv("data/mahindra/soh/rul_baseline.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 5))
for vin, g in model_df.groupby("vin"):
    g = g.sort_values("age_months"); ax.plot(g["age_months"], g["soh"], marker=".", alpha=0.7, label=vin[-6:])
ax.axhline(EOL, ls="--", color="red", alpha=0.6, label="EOL 80%")
ax.set_xlabel("Age (months)"); ax.set_ylabel("SoH (%)"); ax.set_title("Cohort SoH trajectories")
ax.grid(alpha=0.3); ax.legend(ncol=3, fontsize=7); plt.tight_layout(); plt.show()